# Clasificador de intenciones — ClasesPro
## Evaluación del prompt de Gemini para clasificación de mensajes de WhatsApp

**Integrante:** [Tu nombre]  
**Materia:** Inteligencia Artificial  
**Servicio:** Clases universitarias  

### Objetivo
Evaluar la precisión del prompt de Gemini para clasificar mensajes de prospectos
en 6 intenciones: precio, disponibilidad, listo_compra, queja, saludo, fuera_tema
y 3 niveles de interés: frío, tibio, caliente.

In [ ]:
# Instalación de dependencias
!pip install google-genai pandas scikit-learn -q

# Imports
from google import genai
from google.genai import types
import pandas as pd
import json
import os

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [ ]:
# Configura tu API key

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY", "TU_API_KEY_AQUI"))


# El prompt del clasificador
PROMPT_SISTEMA = """Eres el asistente de ventas por WhatsApp de ClasesPro, servicio de clases universitarias en Colombia.
Lee el mensaje y clasificalo. Respondes SIEMPRE en JSON valido sin texto adicional.
intencion: una de precio, disponibilidad, listo_compra, queja, saludo, fuera_tema.
nivel_interes: una de frio, tibio, caliente.
Reglas estrictas:
- Si intencion es listo_compra entonces nivel_interes es caliente.
- Si intencion es queja entonces nivel_interes es frio.
- Si intencion es fuera_tema o saludo entonces nivel_interes es frio.
- Si intencion es precio o disponibilidad y el cliente muestra urgencia o dice que ya decidio entonces nivel_interes es caliente, si no es tibio.
Devuelve solo: {"intencion":"<valor>","nivel_interes":"<valor>"}"""

print("✅ Configuración lista")

✅ Configuración lista


In [ ]:
datos = [
    # precio - tibio (interés moderado)
    {"mensaje": "Cuánto cuesta una hora de clase de estadística?", "intencion_real": "precio", "nivel_real": "tibio"},
    {"mensaje": "Qué planes tienen y cuánto cuestan?", "intencion_real": "precio", "nivel_real": "tibio"},
    {"mensaje": "Me puede decir los precios de las clases?", "intencion_real": "precio", "nivel_real": "tibio"},

    # precio - caliente (ya casi decide)
    {"mensaje": "Cuánto cuesta? me interesa empezar esta semana", "intencion_real": "precio", "nivel_real": "caliente"},
    {"mensaje": "Vi sus clases y me convencieron, cuánto es el paquete completo?", "intencion_real": "precio", "nivel_real": "caliente"},

    # disponibilidad - tibio
    {"mensaje": "Tienen clases los fines de semana?", "intencion_real": "disponibilidad", "nivel_real": "tibio"},
    {"mensaje": "Hay horario disponible para mañana?", "intencion_real": "disponibilidad", "nivel_real": "tibio"},
    {"mensaje": "Puedo agendar una clase para el viernes?", "intencion_real": "disponibilidad", "nivel_real": "tibio"},

    # listo_compra - caliente
    {"mensaje": "Quiero contratar las clases de cálculo, cómo pago?", "intencion_real": "listo_compra", "nivel_real": "caliente"},
    {"mensaje": "Ya decidí, quiero empezar esta semana", "intencion_real": "listo_compra", "nivel_real": "caliente"},
    {"mensaje": "Me interesa el paquete completo, cuál es el número de cuenta?", "intencion_real": "listo_compra", "nivel_real": "caliente"},

    # queja - frio
    {"mensaje": "El profesor llegó tarde a la clase de ayer", "intencion_real": "queja", "nivel_real": "frio"},
    {"mensaje": "No me gustó la clase, no entendí nada", "intencion_real": "queja", "nivel_real": "frio"},
    {"mensaje": "Llevo dos días esperando respuesta y nadie me contesta", "intencion_real": "queja", "nivel_real": "frio"},

    # saludo - frio
    {"mensaje": "Hola buenas tardes", "intencion_real": "saludo", "nivel_real": "frio"},
    {"mensaje": "Hola!", "intencion_real": "saludo", "nivel_real": "frio"},

    # fuera_tema - frio
    {"mensaje": "Cuál es la receta de la bandeja paisa?", "intencion_real": "fuera_tema", "nivel_real": "frio"},
    {"mensaje": "Me recomiendas una película?", "intencion_real": "fuera_tema", "nivel_real": "frio"},
    {"mensaje": "Quién ganó el partido de anoche?", "intencion_real": "fuera_tema", "nivel_real": "frio"},
    {"mensaje": "Cómo está el clima hoy?", "intencion_real": "fuera_tema", "nivel_real": "frio"},
]

df = pd.DataFrame(datos)
print(f"✅ Dataset creado con {len(df)} mensajes")
df.head()

✅ Dataset creado con 20 mensajes


,mensaje,intencion_real,nivel_real
0,Cuánto cuesta una hora de clase de estadística?,precio,tibio
1,Qué planes tienen y cuánto cuestan?,precio,tibio
2,Me puede decir los precios de las clases?,precio,tibio
3,Cuánto cuesta? me interesa empezar esta semana,precio,caliente
4,"Vi sus clases y me convencieron, cuánto es el ...",precio,caliente


In [ ]:
# Función para clasificar un mensaje con Gemini
def clasificar_mensaje(mensaje):
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            config=types.GenerateContentConfig(
                system_instruction=PROMPT_SISTEMA,
                temperature=0,
                response_mime_type="application/json"
            ),
            contents=mensaje
        )
        resultado = json.loads(response.text)
        return resultado.get("intencion", "error"), resultado.get("nivel_interes", "error")
    except Exception as e:
        print(f"Error: {e}")
        return "error", "error"

print("✅ Función lista")

# Clasificar todos los mensajes del dataset
print("Clasificando mensajes con Gemini...")
intenciones_pred = []
niveles_pred = []

for i, row in df.iterrows():
    intencion, nivel = clasificar_mensaje(row["mensaje"])
    intenciones_pred.append(intencion)
    niveles_pred.append(nivel)
    print(f"[{i+1}/20] '{row['mensaje'][:35]}...' → {intencion} | {nivel}")

df["intencion_pred"] = intenciones_pred
df["nivel_pred"] = niveles_pred
print("\n✅ Clasificación completa")

✅ Función lista
Clasificando mensajes con Gemini...
[1/20] 'Cuánto cuesta una hora de clase de ...' → precio | tibio
[2/20] 'Qué planes tienen y cuánto cuestan?...' → precio | tibio
[3/20] 'Me puede decir los precios de las c...' → precio | tibio
[4/20] 'Cuánto cuesta? me interesa empezar ...' → precio | caliente
[5/20] 'Vi sus clases y me convencieron, cu...' → precio | caliente
[6/20] 'Tienen clases los fines de semana?...' → disponibilidad | tibio
[7/20] 'Hay horario disponible para mañana?...' → disponibilidad | tibio
[8/20] 'Puedo agendar una clase para el vie...' → disponibilidad | tibio
[9/20] 'Quiero contratar las clases de cálc...' → listo_compra | caliente
[10/20] 'Ya decidí, quiero empezar esta sema...' → listo_compra | caliente
[11/20] 'Me interesa el paquete completo, cu...' → listo_compra | caliente
[12/20] 'El profesor llegó tarde a la clase ...' → queja | frio
[13/20] 'No me gustó la clase, no entendí na...' → queja | frio
[14/20] 'Llevo dos días esperando respuesta ...

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Accuracy de intención
acc_intencion = accuracy_score(df["intencion_real"], df["intencion_pred"])

# Accuracy de nivel de interés
acc_nivel = accuracy_score(df["nivel_real"], df["nivel_pred"])

print("="*50)
print("📊 MÉTRICAS DEL CLASIFICADOR — ClasesPro")
print("="*50)
print(f"\n✅ Accuracy intención:      {acc_intencion*100:.1f}%")
print(f"✅ Accuracy nivel interés:  {acc_nivel*100:.1f}%")

print("\n--- Reporte detallado de intención ---")
print(classification_report(df["intencion_real"], df["intencion_pred"]))

print("\n--- Reporte detallado de nivel de interés ---")
print(classification_report(df["nivel_real"], df["nivel_pred"]))

📊 MÉTRICAS DEL CLASIFICADOR — ClasesPro

✅ Accuracy intención:      100.0%
✅ Accuracy nivel interés:  100.0%

--- Reporte detallado de intención ---
                precision    recall  f1-score   support

disponibilidad       1.00      1.00      1.00         3
    fuera_tema       1.00      1.00      1.00         4
  listo_compra       1.00      1.00      1.00         3
        precio       1.00      1.00      1.00         5
         queja       1.00      1.00      1.00         3
        saludo       1.00      1.00      1.00         2

      accuracy                           1.00        20
     macro avg       1.00      1.00      1.00        20
  weighted avg       1.00      1.00      1.00        20


--- Reporte detallado de nivel de interés ---
              precision    recall  f1-score   support

    caliente       1.00      1.00      1.00         5
        frio       1.00      1.00      1.00         9
       tibio       1.00      1.00      1.00         6

    accuracy           

In [ ]:
# 3 ejemplos comentados para el informe
print("="*50)
print("📝 EJEMPLOS COMENTADOS")
print("="*50)

ejemplos = [
    {
        "mensaje": "Quiero contratar las clases de cálculo, cómo pago?",
        "esperado_intencion": "listo_compra",
        "esperado_nivel": "caliente",
        "comentario": "Mensaje claro de compra inmediata. El cliente ya decidió y pregunta por el método de pago. Gemini debe detectar listo_compra y nivel caliente correctamente."
    },
    {
        "mensaje": "Cuánto cuesta una hora de clase de estadística?",
        "esperado_intencion": "precio",
        "esperado_nivel": "tibio",
        "comentario": "Consulta directa de precio. El cliente tiene interés pero aún no decide. Se espera intención precio y nivel tibio."
    },
    {
        "mensaje": "Cuál es la receta de la bandeja paisa?",
        "esperado_intencion": "fuera_tema",
        "esperado_nivel": "frio",
        "comentario": "Mensaje completamente ajeno al servicio. Gemini debe identificarlo como fuera_tema y nivel frio sin confundirse."
    }
]

for i, ej in enumerate(ejemplos, 1):
    intencion_pred, nivel_pred = clasificar_mensaje(ej["mensaje"])
    correcto = "✅" if intencion_pred == ej["esperado_intencion"] else "❌"

    print(f"\nEjemplo {i}: {correcto}")
    print(f"  Mensaje:    '{ej['mensaje']}'")
    print(f"  Esperado:   {ej['esperado_intencion']} | {ej['esperado_nivel']}")
    print(f"  Predicho:   {intencion_pred} | {nivel_pred}")
    print(f"  Análisis:   {ej['comentario']}")

print("\n" + "="*50)
print("✅ Evaluación completa del clasificador")
print("="*50)

📝 EJEMPLOS COMENTADOS

Ejemplo 1: ✅
  Mensaje:    'Quiero contratar las clases de cálculo, cómo pago?'
  Esperado:   listo_compra | caliente
  Predicho:   listo_compra | caliente
  Análisis:   Mensaje claro de compra inmediata. El cliente ya decidió y pregunta por el método de pago. Gemini debe detectar listo_compra y nivel caliente correctamente.

Ejemplo 2: ✅
  Mensaje:    'Cuánto cuesta una hora de clase de estadística?'
  Esperado:   precio | tibio
  Predicho:   precio | tibio
  Análisis:   Consulta directa de precio. El cliente tiene interés pero aún no decide. Se espera intención precio y nivel tibio.

Ejemplo 3: ✅
  Mensaje:    'Cuál es la receta de la bandeja paisa?'
  Esperado:   fuera_tema | frio
  Predicho:   fuera_tema | frio
  Análisis:   Mensaje completamente ajeno al servicio. Gemini debe identificarlo como fuera_tema y nivel frio sin confundirse.

✅ Evaluación completa del clasificador
